# TDT4173 Hydropower unit commitment: Short notebook 1

**Group number:** [FILL IN]
**Kaggle team name:** [FILL IN, including the group number]
**Members (full name, student ID):**
- [Full name 1], [student ID]
- [Full name 2], [student ID]
- [Full name 3], [student ID]
- [Full name 4], [student ID]
- [Full name 5], [student ID]

Reproduces our selected Kaggle prediction from the raw supplied data, offline. Output: `submission.csv`.

**Method** (details in the report): a simplified physics model of the Tokke–Vinje system is solved for every week, as a linear program (10 water-value variants) and as a unit-commitment MILP (on/off per generator, 3 water-value scales). Its schedules are features for one XGBoost per unit and for two sequence networks (CNN+GRU and transformer). The three models are blended in log-odds space (0.75 / 0.20 / 0.05). The XGBoost settings, tree counts and blend weights were chosen in our validation experiments (early stopping on 2021, blend search on the 2022 hold-out) and are fixed here.

**Hardware and run time.** Runs on CPU only or with an NVIDIA GPU (used automatically if PyTorch sees one; we used an NVIDIA L4, 24 GB, CUDA). Expected time on a 4-core PC: about 3 to 4 hours with a mid-range GPU, about 7 to 9 hours CPU only. Peak memory about 8 to 12 GB. The notebook prints the elapsed time after each stage.

**Packages:** see `requirements.txt` (numpy, pandas 2.2.3, scipy ≥ 1.11, scikit-learn, pyyaml, joblib, xgboost ≥ 2.0, torch).

Place this notebook in the folder that contains `data/kernel/`, `data/extended/`, `sample_submission.csv` and `prediction_mapping.csv`, then run all cells from top to bottom. Intermediate results are saved in `outputs/`; if the run is interrupted, running it again continues where it stopped.

## Settings

In [1]:
# ============================  SETTINGS  ============================
# Light version of the RunPod pipeline that produced Kaggle 0.99512, sized to finish within 12 hours on a
# normal PC (4 CPU cores, 32 GB RAM; a mid-range NVIDIA GPU makes it much faster but is not required).
# The per-unit XGBoost settings and tree counts, and the blend weights, were chosen in the full run
# (early stopping on 2021 and a search on the 2022 hold-out; see the report) and are fixed here.
import os, time
from pathlib import Path
T_START = time.time()
DATA_DIR = Path(".")                        # folder with data/, sample_submission.csv, prediction_mapping.csv
DRY_RUN = os.environ.get("DRY_RUN", "0") == "1"   # optional quick test on ~150 weeks (a few minutes)
N_CPU = os.cpu_count() or 1

# physics simulation (LP) variants: (name, water-value multiplier, how water lost via bypass/spill is valued)
LP_VARIANTS = [("base", 1.0, "local"), ("v2g", 1.0, "global"), ("s09", 0.9, "local"), ("s11", 1.1, "local"),
               ("s08", 0.8, "local"), ("s095", 0.95, "local"), ("s105", 1.05, "local"), ("s12", 1.2, "local"),
               ("g09", 0.9, "global"), ("g11", 1.1, "global")]
# unit-commitment MILP (on/off per generator, p_min, start/stop costs)
RUN_UC_MIP = True
UC_SCALES = {"uc10": 1.0, "uc09": 0.9, "uc11": 1.1}   # water-value multipliers for the MILP runs
UC_NODE_LIMIT = 50           # branch-and-bound node limit per week (full run: 500)
UC_TIME_LIMIT = 30           # safety limit in seconds per week; the node limit normally stops first
UC_GAP = 0.005
# XGBoost: settings chosen per unit by early stopping on 2021 in the full run -> (setting, best tree count)
XGB_CONFIGS = {"d4": dict(max_depth=4, learning_rate=0.05, min_child_weight=10),
               "d6": dict(max_depth=6, learning_rate=0.03, min_child_weight=20)}
XGB_FIXED = {"Hogga_G1": ("d6", 1939), "Tokke_G1": ("d6", 1074), "Tokke_G2": ("d6", 629), "Tokke_G3": ("d6", 1333),
             "Tokke_G4": ("d4", 834), "Lio_G1": ("d6", 836), "Byrte_G1": ("d6", 574), "Vinje_G1": ("d6", 1027),
             "Vinje_G2": ("d6", 706), "Vinje_G3": ("d6", 713), "Haukeli_G1": ("d6", 1416), "Songa_G1": ("d6", 871),
             "Kjela_G1": ("d6", 1281), "Vesle Kjela_G1": ("d6", 1058)}
XGB_SEEDS = 2                # full run: 3
# sequence networks
NN_TYPES = ["cnn_gru", "transformer"]
NN_SEEDS = {"cnn_gru": 3, "transformer": 1}            # full run: 5 each
NN_EPOCHS = 40
# blend weights in log-odds space, chosen on the 2022 hold-out in the full run
BLEND_W = {"xgb": 0.75, "nn_cnn_gru": 0.20, "nn_transformer": 0.05}

if DRY_RUN:
    UC_NODE_LIMIT, UC_TIME_LIMIT = 20, 10
    XGB_FIXED = {u: (c, min(n, 60)) for u, (c, n) in XGB_FIXED.items()}
    XGB_SEEDS, NN_EPOCHS = 1, 2
    NN_SEEDS = {k: 1 for k in NN_SEEDS}
RESULTS = DATA_DIR / ("outputs_dry" if DRY_RUN else "outputs")
RESULTS.mkdir(exist_ok=True)
assert (DATA_DIR / "sample_submission.csv").exists(), "put the notebook in the folder with data/ and sample_submission.csv"
print("DRY_RUN =", DRY_RUN, "| CPU cores:", N_CPU, "| results ->", RESULTS)

DRY_RUN = False | CPU cores: 12 | results -> outputs


## Feature engineering (group feature table)

In [2]:
from pathlib import Path
import json
import hashlib
import platform
import time
import warnings
import numpy as np
import pandas as pd
import scipy
import sklearn
import yaml
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# (DATA_DIR is set in the settings cell)
OUT = DATA_DIR / "feature_preprocessing_outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
HORIZON = 168
EVALUATION_START = pd.Timestamp("2022-01-01", tz="UTC")
INNER_YEARS = (2020, 2021)
CASE_STRIDE_DAYS = 8
MI_SAMPLE_ROWS = 2048
PERMUTATION_REPEATS = 2
SHORTLIST_SIZE = 24
MAX_REDUCED_FEATURES = 48
MIN_IMPORTANCE = 0.0005  # Practical screening threshold, not a significance cutoff.
MAX_ALLOWED_AUC_LOSS = 0.002  # Retain broad set if reduction hurts more on either fold.
PROBE_PARAMS = dict(n_estimators=64, max_depth=10, min_samples_leaf=20,
                    max_features=0.7, n_jobs=4, random_state=SEED)
versions = dict(python=platform.python_version(), numpy=np.__version__,
                pandas=pd.__version__, scipy=scipy.__version__, sklearn=sklearn.__version__,
                pyyaml=yaml.__version__)
print("Versions:", versions)


raw_audit = []
source_paths = []
def read_series(relative_path):
    path = DATA_DIR / relative_path
    source_paths.append(path)
    frame = pd.read_csv(path)
    frame["date"] = pd.to_datetime(frame["date"], utc=True)
    assert frame["date"].is_unique, f"Duplicate dates: {path.name}"
    frame = frame.set_index("date").sort_index()
    frame = frame.apply(pd.to_numeric, errors="raise").astype("float32")
    assert not np.isinf(frame.to_numpy()).any(), f"Infinite values: {path.name}"
    raw_audit.append(dict(file=path.name, rows=len(frame), columns=len(frame.columns),
                          first=str(frame.index.min()), last=str(frame.index.max()),
                          missing_values=int(frame.isna().sum().sum()),
                          negative_values=int((frame < 0).sum().sum())))
    return frame

price = read_series("data/kernel/Historical_day_ahead_price_2015_2025.csv")
inflow = read_series("data/kernel/Historical_inflow_1958_2025.csv")
storage = read_series("data/kernel/Historical_volume_2015_2024.csv")
water = read_series("data/kernel/Synthetic_water_value_2015_2024.csv")
min_volume = read_series("data/extended/Constraint_min_volume.csv")
min_flow = read_series("data/extended/Constraint_min_flow.csv")
topology_path = DATA_DIR / "data/extended/Tokke_Vinje_topology.yaml"
topology = yaml.safe_load(topology_path.read_text(encoding="utf-8"))
source_paths.append(topology_path)
decision_path = DATA_DIR / "data/kernel/Unit_commitment_decisions.csv"
decisions = pd.read_csv(decision_path)
source_paths.append(decision_path)
decisions["starttime"] = pd.to_datetime(decisions["starttime"], utc=True)
decisions = decisions.sort_values("starttime").reset_index(drop=True)
assert decisions["Run No"].is_unique and decisions["starttime"].is_unique
mapping = pd.read_csv(DATA_DIR / "prediction_mapping.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv", nrows=0)
source_paths.append(DATA_DIR / "prediction_mapping.csv")
mapping["starttime"] = pd.to_datetime(mapping["starttime"], utc=True)
assert mapping["Run No"].is_unique
assert not set(mapping["Run No"]) & set(decisions["Run No"])
unit_columns = {}
for col in decisions.columns:
    if col.startswith("result_committed_"):
        unit = col.removeprefix("result_committed_").rsplit("_t", 1)[0]
        unit_columns.setdefault(unit, []).append(col)
for unit, cols in unit_columns.items():
    unit_columns[unit] = sorted(cols, key=lambda c: int(c.rsplit("_t", 1)[1]))
    assert [int(c.rsplit("_t", 1)[1]) for c in unit_columns[unit]] == list(range(HORIZON))
assert len(unit_columns) == 14
assert set(sample.columns) - {"Run No"} == {c for cols in unit_columns.values() for c in cols}
assert np.isin(decisions[[c for cols in unit_columns.values() for c in cols]].to_numpy(), [0, 1]).all()
pd.DataFrame(raw_audit).to_csv(OUT / "01_raw_data_audit.csv", index=False)
print(pd.DataFrame(raw_audit).to_string(index=False))
print("Labelled cases:", len(decisions), "Kaggle cases:", len(mapping))


cases = pd.concat([decisions[["Run No", "starttime"]].assign(source="labelled"),
                   mapping[["Run No", "starttime"]].assign(source="kaggle")], ignore_index=True)
starts = pd.DatetimeIndex(cases["starttime"])
ends = starts + pd.Timedelta(hours=HORIZON - 1)
labelled = cases["source"].eq("labelled").to_numpy()
dev_cases = np.flatnonzero(labelled & (ends < EVALUATION_START))
eval_cases = np.flatnonzero(labelled & (starts >= EVALUATION_START))
test_cases = np.flatnonzero(~labelled)
assert ends[dev_cases].max() < starts[eval_cases].min()
day_offsets = ((starts - starts.min()) / pd.Timedelta(days=1)).to_numpy()
weekly = (day_offsets % CASE_STRIDE_DAYS) == 0
folds = []
for year in INNER_YEARS:
    boundary = pd.Timestamp(f"{year}-01-01", tz="UTC")
    stop = pd.Timestamp(f"{year+1}-01-01", tz="UTC")
    tr = np.flatnonzero(labelled & weekly & (ends < boundary))
    va = np.flatnonzero(labelled & weekly & (starts >= boundary) & (ends < stop))
    assert ends[tr].max() < starts[va].min()
    assert np.all(np.diff(starts[tr].asi8) >= pd.Timedelta(days=7).value)
    assert np.all(np.diff(starts[va].asi8) >= pd.Timedelta(days=7).value)
    assert ends[va].max() < EVALUATION_START
    folds.append(dict(year=year, train=tr, valid=va))
cases["partition"] = "boundary_excluded"
cases.loc[dev_cases, "partition"] = "development"
cases.loc[eval_cases, "partition"] = "evaluation_2022"
cases.loc[test_cases, "partition"] = "kaggle"
cases.to_csv(OUT / "02_case_manifest.csv", index=False)
print(cases.groupby("partition").size().to_string())
for f in folds:
    print(f["year"], "screening train/validation cases:", len(f["train"]), len(f["valid"]))


reservoirs = list(storage.columns)
assert set(water.columns) == set(reservoirs)
water = water[reservoirs]
assert set(reservoirs) <= set(inflow.columns)
capacity = np.array([topology["model"]["reservoir"][r]["max_vol"] for r in reservoirs], dtype="float32")
assert (capacity > 0).all()
def lookup(frame, timestamps):
    timestamps = pd.DatetimeIndex(timestamps)
    missing = timestamps.difference(frame.index)
    assert len(missing) == 0, f"Missing required timestamps, e.g. {missing[:3].tolist()}"
    return frame.reindex(timestamps).to_numpy(dtype="float32")

hour_times = pd.to_datetime((starts.asi8[:, None] +
                            np.arange(HORIZON)[None, :] * pd.Timedelta(hours=1).value).ravel(), utc=True)
P = lookup(price, hour_times).reshape(len(cases), HORIZON)
I = lookup(inflow, hour_times).reshape(len(cases), HORIZON, len(inflow.columns))
S = lookup(storage, starts)
W = lookup(water, starts + pd.Timedelta(days=7))
MV = lookup(min_volume, hour_times).reshape(len(cases), HORIZON, len(min_volume.columns))
MF = lookup(min_flow, hour_times).reshape(len(cases), HORIZON, len(min_flow.columns))
print("Price shape:", P.shape, "Inflow shape:", I.shape)
print("Initial storage outside [0, capacity]:", int(((S < 0) | (S > capacity)).sum()))

edges = topology["connections"]
def nearest_reservoirs(node_type, node_name, direction):
    found, seen, queue = set(), set(), [(node_type, node_name)]
    while queue:
        kind, name = queue.pop()
        if (kind, name) in seen:
            continue
        seen.add((kind, name))
        for e in edges:
            if direction == "up" and e["to"] == name and e["to_type"] == kind:
                next_kind, next_name = e["from_type"], e["from"]
            elif direction == "down" and e["from"] == name and e["from_type"] == kind:
                next_kind, next_name = e["to_type"], e["to"]
            else:
                continue
            if next_kind == "reservoir":
                found.add(next_name)
            elif next_kind in {"river", "tunnel"}:
                queue.append((next_kind, next_name))
    return sorted(found)

connections = {}
for unit in unit_columns:
    plant_edges = [e for e in edges if e["from_type"] == "generator" and e["from"] == unit
                   and e["to_type"] == "plant"]
    assert len(plant_edges) == 1
    plant = plant_edges[0]["to"]
    src = nearest_reservoirs("plant", plant, "up")
    dst = nearest_reservoirs("plant", plant, "down")
    assert src and set(src + dst) <= set(reservoirs)
    connections[unit] = dict(plant=plant, sources=src, destinations=dst)
print(pd.DataFrame(connections).T.to_string())
(OUT / "03_unit_reservoir_connections.json").write_text(json.dumps(connections, indent=2))


registry = {}
feature_arrays = {}
def add(name, values, family, units, meaning):
    a = np.asarray(values, dtype="float32")
    if a.ndim == 1:
        a = a[:, None]
    a = np.broadcast_to(a, (len(cases), HORIZON))
    assert not np.isinf(a).any(), name
    feature_arrays[name] = a.ravel()
    registry[name] = dict(feature=name, family=family, units=units, meaning=meaning)

price_rank = pd.DataFrame(P).rank(axis=1, pct=True, method="average").to_numpy(dtype="float32")
mean_p = np.mean(P, axis=1)
std_p = np.std(P, axis=1)
add("price", P, "price_level", "EUR/MWh", "Price at the horizon hour")
add("price_rank", price_rank, "price_context", "fraction", "Rank across this case's 168 prices")
for name, arr in [("mean", mean_p), ("std", std_p), ("min", P.min(axis=1)), ("max", P.max(axis=1))]:
    add("price_week_" + name, arr, "price_context", "EUR/MWh", "Whole supplied horizon price " + name)
add("price_minus_week_mean", P - mean_p[:, None], "price_context", "EUR/MWh", "Price minus horizon mean")
add("price_within_week_z", (P - mean_p[:, None]) / np.maximum(std_p[:, None], 1),
    "price_context", "ratio", "Price centered/scaled within this case; denominator floor 1 EUR/MWh")
previous = np.column_stack([P[:, 0], P[:, :-1]])
following = np.column_stack([P[:, 1:], P[:, -1]])
add("price_change_previous", P - previous, "price_context", "EUR/MWh", "Current minus previous horizon price; zero at t0")
add("price_change_next", following - P, "price_context", "EUR/MWh", "Next minus current horizon price; zero at t167")
for width in [6, 24]:
    future_mean = np.column_stack([P[:, h:min(HORIZON, h+width)].mean(axis=1) for h in range(HORIZON)])
    add(f"price_minus_next_{width}h_mean", P - future_mean, "price_context", "EUR/MWh",
        f"Current price minus mean of current and up to {width-1} following supplied hours")
remaining_mean = np.flip(np.cumsum(np.flip(P, axis=1), axis=1), axis=1) / np.arange(HORIZON, 0, -1)
add("price_minus_remaining_mean", P - remaining_mean, "price_context", "EUR/MWh", "Current minus mean from current hour to horizon end")
h = np.broadcast_to(np.arange(HORIZON), P.shape)
add("horizon_fraction", h / (HORIZON - 1), "calendar", "fraction", "Position within the scheduling horizon")
add("first_horizon_hour", h == 0, "calendar", "binary", "No previous price inside case")
add("last_horizon_hour", h == HORIZON - 1, "calendar", "binary", "No following price inside case")
for name, values, period in [("hour", hour_times.hour.to_numpy().reshape(P.shape), 24),
                             ("weekday", hour_times.dayofweek.to_numpy().reshape(P.shape), 7),
                             ("season", hour_times.dayofyear.to_numpy().reshape(P.shape)-1, 365.25)]:
    add(name + "_sin", np.sin(2*np.pi*values/period), "calendar", "unitless", "Cyclic " + name)
    add(name + "_cos", np.cos(2*np.pi*values/period), "calendar", "unitless", "Cyclic " + name)

for j, r in enumerate(reservoirs):
    add("storage_fraction__" + r, S[:, j]/capacity[j], "storage", "fraction", r + " starting volume / configured max volume")
    add("water_value__" + r, W[:, j], "water_value", "EUR/MWh", r + " water value at start + 7 days")
    margin = P - W[:, j, None]
    add("price_margin__" + r, margin, "price_water_margin", "EUR/MWh", "Price minus terminal water value of " + r)
    add("relative_margin__" + r, margin/(np.abs(P)+np.abs(W[:, j, None])+1),
        "price_water_margin", "ratio", "Bounded relative price-water-value margin for " + r)
for j, r in enumerate(inflow.columns):
    q = I[:, :, j]
    cum = np.cumsum(q, axis=1) * 0.0036
    total = np.sum(q, axis=1) * 0.0036
    add("inflow_now__" + r, q, "inflow", "m3/s", "Supplied hourly inflow into " + r)
    add("inflow_week__" + r, total, "inflow", "million m3", "Total supplied horizon inflow into " + r)
    add("inflow_cumulative__" + r, cum, "inflow", "million m3", "Inflow through and including current hour into " + r)
    add("inflow_remaining__" + r, total[:, None]-cum, "inflow", "million m3", "Inflow after current hour into " + r)
for j, r in enumerate(min_volume.columns):
    idx = reservoirs.index(r)
    add("min_storage_fraction__" + r, MV[:, :, j]/capacity[idx], "constraints", "fraction", "Required minimum volume / capacity")
    add("initial_storage_above_min__" + r, (S[:, idx, None]-MV[:, :, j])/capacity[idx],
        "constraints", "fraction", "INITIAL storage minus this hour's minimum, divided by capacity; not actual future storage")
for j, r in enumerate(min_flow.columns):
    add("min_flow__" + r, MF[:, :, j], "constraints", "m3/s", "Required minimum river flow: " + r)
shared = pd.DataFrame(feature_arrays, dtype="float32")
del feature_arrays
assert len(shared) == len(cases) * HORIZON
print("Shared feature matrix:", shared.shape, "Memory MB:", round(shared.memory_usage().sum()/1e6, 1))


def row_indices(case_indices):
    return (np.asarray(case_indices)[:, None]*HORIZON + np.arange(HORIZON)[None, :]).ravel()

def build_unit_table(unit, case_indices):
    ci = np.asarray(case_indices)
    info = connections[unit]
    sj = [reservoirs.index(r) for r in info["sources"]]
    ij = [list(inflow.columns).index(r) for r in info["sources"]]
    total_capacity = capacity[sj].sum()
    initial = S[ci][:, sj].sum(axis=1)
    wv = (W[ci][:, sj]*capacity[sj]).sum(axis=1)/total_capacity
    q = I[ci][:, :, ij].sum(axis=2)
    cum = np.cumsum(q, axis=1)*0.0036
    total = q.sum(axis=1)*0.0036
    p = P[ci]
    local = {}
    def put(name, value, units, meaning):
        a = np.asarray(value, dtype="float32")
        if a.ndim == 1:
            a = a[:, None]
        local[name] = np.broadcast_to(a, p.shape).ravel()
        registry[name] = dict(feature=name, family="local_hydrology", units=units, meaning=meaning)
    put("source_storage_fraction", initial/total_capacity, "fraction", "Combined initial source storage / combined capacity")
    put("source_water_value", wv, "EUR/MWh", "Capacity-weighted source terminal water value; descriptive proxy")
    put("source_price_margin", p-wv[:, None], "EUR/MWh", "Price minus capacity-weighted source water value; proxy")
    put("source_relative_margin", (p-wv[:, None])/(np.abs(p)+np.abs(wv[:, None])+1), "ratio", "Bounded relative source price margin")
    put("source_inflow_now", q, "m3/s", "Combined natural inflow to immediate source reservoirs")
    put("source_inflow_week_fraction", total/total_capacity, "fraction", "Horizon natural inflow / combined source capacity")
    put("source_inflow_cumulative_fraction", cum/total_capacity, "fraction", "Natural inflow through current hour / capacity")
    put("source_inflow_remaining_fraction", (total[:, None]-cum)/total_capacity, "fraction", "Natural inflow after current hour / capacity")
    put("source_no_release_fill_fraction", (initial[:, None]+cum)/total_capacity, "fraction", "Initial volume + cumulative natural inflow, ignoring all releases/transfers")
    put("source_week_inflow_minus_headroom", (initial+total-total_capacity)/total_capacity, "fraction", "No-release horizon volume minus capacity, normalized")
    put("source_margin_times_storage", (p-wv[:, None])*(initial/total_capacity)[:, None], "EUR/MWh", "Explicit price-margin x initial-storage interaction")
    if info["destinations"]:
        dj = [reservoirs.index(r) for r in info["destinations"]]
        dc = capacity[dj].sum()
        dw = (W[ci][:, dj]*capacity[dj]).sum(axis=1)/dc
        put("destination_storage_fraction", S[ci][:, dj].sum(axis=1)/dc, "fraction", "Combined initial downstream reservoir storage / capacity")
        put("source_minus_destination_water_value", wv-dw, "EUR/MWh", "Descriptive water-value contrast; NOT an exact cascade opportunity cost")
    X = pd.concat([shared.iloc[row_indices(ci)].reset_index(drop=True), pd.DataFrame(local)], axis=1)
    assert X.columns.is_unique and X.shape[0] == len(ci)*HORIZON
    assert not np.isinf(X.to_numpy()).any()
    return X

def labels_for(unit, case_indices):
    assert np.asarray(case_indices).max() < len(decisions), "Kaggle cases have no labels"
    return decisions.iloc[case_indices][unit_columns[unit]].to_numpy(dtype="uint8").ravel()

example_unit = next(iter(unit_columns))
example = build_unit_table(example_unit, [0, 1])
assert np.allclose(example["price"].to_numpy().reshape(2, HORIZON), P[:2], equal_nan=True)
assert np.array_equal(labels_for(example_unit, [0, 1]).reshape(2, HORIZON),
                      decisions.iloc[:2][unit_columns[example_unit]].to_numpy())
assert np.allclose(shared["inflow_remaining__"+inflow.columns[0]].to_numpy().reshape(P.shape)[:, -1], 0,
                   atol=0.01, equal_nan=True)
pd.DataFrame(registry.values()).to_csv(OUT / "04_feature_dictionary.csv", index=False)
print("Example unit:", example_unit, "candidate features:", example.shape[1])
print(example[["price", "price_rank", "source_storage_fraction", "source_price_margin"]].head().to_string(index=False))

Versions: {'python': '3.11.5', 'numpy': '2.2.6', 'pandas': '2.2.3', 'scipy': '1.15.3', 'sklearn': '1.6.1', 'pyyaml': '6.0.3'}
                                    file   rows  columns                     first                      last  missing_values  negative_values
Historical_day_ahead_price_2015_2025.csv 103062        1 2015-01-01 00:00:00+00:00 2025-12-31 23:45:00+00:00               0              449
         Historical_inflow_1958_2025.csv 594224       18 1958-01-01 00:00:00+00:00 2025-10-15 07:00:00+00:00               0                0
         Historical_volume_2015_2024.csv   3654       17 2015-01-01 00:00:00+00:00 2025-01-01 00:00:00+00:00               0                0
     Synthetic_water_value_2015_2024.csv   3654       17 2015-01-01 00:00:00+00:00 2025-01-01 00:00:00+00:00               0                0
               Constraint_min_volume.csv  96432        3 2015-01-01 00:00:00+00:00 2025-12-31 23:00:00+00:00               0                0
                 Const

## Shared setup

In [3]:
import time, gc, glob
import numpy as np, pandas as pd, yaml, scipy.sparse as sp
import torch, torch.nn as nn
from xgboost import XGBClassifier
from scipy.optimize import linprog, milp, LinearConstraint, Bounds
from scipy.ndimage import uniform_filter1d
from scipy.special import logit, expit
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
T = 168
UNITS = list(unit_columns)                       # 14 generators, same order as the submission columns
NC = len(cases); NTR = int(labelled.sum())
assert labelled[:NTR].all() and not labelled[NTR:].any()
RUN_NO = cases["Run No"].to_numpy()
Y = np.stack([decisions[unit_columns[u]].to_numpy(np.uint8) for u in UNITS], 1)       # (2922, 14, 168)
assert (decisions["Run No"].to_numpy() == RUN_NO[:NTR]).all()
W0 = lookup(water, starts)                       # water value at the start of each week
START_H = starts.tz_convert(None).values.astype("datetime64[h]").astype(np.int64)
INFLOW_COLS, MV_COLS, MF_COLS = list(inflow.columns), list(min_volume.columns), list(min_flow.columns)
PLANT_OF = {u: connections[u]["plant"] for u in UNITS}

# splits (all purged: a training week must end before the first validation week starts)
Y2021 = pd.Timestamp("2021-01-01", tz="UTC")
SEL = np.arange(NC)[::25] if DRY_RUN else np.arange(NC)
keep = lambda a: np.intersect1d(np.asarray(a), SEL)
ES_TR = keep(np.flatnonzero(labelled & (ends < Y2021)))                       # early stopping: train
ES_VA = keep(np.flatnonzero(labelled & (starts >= Y2021) & (ends < EVALUATION_START)))   # early stopping: 2021
DEV, EVAL = keep(dev_cases), keep(eval_cases)                                  # 2022 hold-out
TRAIN_ALL, TEST = keep(np.flatnonzero(labelled)), keep(np.flatnonzero(~labelled))
print({k: len(v) for k, v in dict(ES_TR=ES_TR, ES_VA=ES_VA, DEV=DEV, EVAL_2022=EVAL, TRAIN_ALL=TRAIN_ALL, TEST=TEST).items()})
rows_of = lambda ci: (np.asarray(ci)[:, None] * T + np.arange(T)[None, :]).ravel()

device: cpu 
{'ES_TR': 2186, 'ES_VA': 359, 'DEV': 2551, 'EVAL_2022': 365, 'TRAIN_ALL': 2922, 'TEST': 725}


## Physics model: LP and unit-commitment MILP

In [4]:
# Physics model of the Tokke-Vinje system, built from the topology YAML.
#  solve():    linear program (LP), plant level, no on/off  -> the features that gave Kaggle 0.99425
#  solve_uc(): unit-commitment MILP on top of the same water network: one on/off variable per generator
#              and hour, p_min/p_max, start and stop costs from the YAML (the decision SHOP actually makes)
DT = 0.0036          # 1 m3/s during 1 hour = 0.0036 Mm3
ETA = 0.88           # turbine x generator efficiency
PEN = 1e4            # penalty per m3/s-hour of unmet minimum flow
CHAIN = {"Staavatn": "Kjelavatn", "Langesae": "Foersvatn", "Langeidvatn": "Vatjern", "Hyljelihyl": "Venemo",
         "Venemo": "Totak", "Bordalsvatn": "Foersvatn", "Totak": "Vaamarvatn"}   # reservoirs without own plant


class System:
    def __init__(s, yaml_path, res_names, inflow_cols, mv_cols, mf_cols):
        y = yaml.safe_load(open(yaml_path)); m = y["model"]; C = y["connections"]
        s.res = list(res_names); s.R = {r: i for i, r in enumerate(s.res)}
        s.maxv = np.array([m["reservoir"][r]["max_vol"] for r in s.res])
        s.vh = {r: (np.array(m["reservoir"][r]["vol_head"]["x"], float), np.array(m["reservoir"][r]["vol_head"]["y"], float)) for r in s.res}
        out = {}
        for c in C:
            out.setdefault(c["from"], []).append((c["to"], c["to_type"]))
        s.out = out

        def dest(obj):          # next reservoir downstream of an object (None = leaves the system)
            for to, tt in out.get(obj, []):
                if tt == "reservoir": return to
                if tt == "river": return dest(to)
            return None
        gens = {}
        for c in C:
            if c["from_type"] == "generator": gens.setdefault(c["to"], []).append(c["from"])
        s.gen = {g: dict(plant=p, pmin=float(m["generator"][g]["p_min"]), pmax=float(m["generator"][g]["p_max"]),
                         sc=float(m["generator"][g].get("startcost_const", 0)), stc=float(m["generator"][g].get("stopcost_const", 0)))
                 for p, gl in gens.items() for g in gl}
        s.plants = {}
        for c in C:
            if c["to_type"] == "plant" and c["from_type"] == "reservoir":
                s.plants.setdefault(c["to"], {"src": []})["src"].append(c["from"])
            if c["to_type"] == "plant" and c["from_type"] == "tunnel":
                src = [cc["from"] for cc in C if cc["to"] == c["from"] and cc["from_type"] == "reservoir"]
                s.plants.setdefault(c["to"], {"src": []})["src"] += src
        for p, d in s.plants.items():
            d["dst"] = dest(p); d["outlet"] = m["plant"][p]["outlet_line"]
            d["pmax"] = sum(m["generator"][g]["p_max"] for g in gens[p])
        s.plist = list(s.plants)
        s.flows = []            # (name, source, destination, cost, lower, upper)
        for c in C:
            if c["from_type"] == "reservoir" and c["to_type"] == "river":
                rv = c["to"]; info = m["river"][rv]; d = dest(rv)
                if rv.startswith("r_"):      # regulated river with increasing cost curve -> 2 convex pieces
                    s.flows.append((rv + "#1", c["from"], d, 1.6, 0.0, 40.0))
                    s.flows.append((rv + "#2", c["from"], d, 3.3, 0.0, 1000.0))
                else:                         # b_ = bypass gate (cost 10), s_ = spill (cost 1000)
                    s.flows.append((rv, c["from"], d, float(info.get("flow_cost_const", 10)), 0.0, 1000.0))
            if c["from_type"] == "reservoir" and c["to_type"] == "tunnel":
                tgt = [to for to, tt in out.get(c["to"], []) if tt == "reservoir"]
                if tgt:                       # free transfer tunnel, both directions
                    s.flows.append((c["to"], c["from"], tgt[0], 0.01, -300.0, 300.0))
        s.inflow_idx = [list(inflow_cols).index(r) for r in s.res]
        s.rvest = list(inflow_cols).index("r_Vest_Vassdraget")
        s.mv_cols = list(mv_cols); s.mf_cols = list(mf_cols)

    def level(s, r, v):
        x, y = s.vh[r]; return float(np.interp(v, x, y))

    def efactor(s, V0):
        e = {}
        for p in s.plist:
            d = s.plants[p]
            for src in d["src"]:
                e[src] = 9.81e-3 * max(s.level(src, V0[s.R[src]]) - d["outlet"], 1.0) * ETA
        for _ in range(5):
            for r, d in CHAIN.items():
                if r not in e and d in e: e[r] = e[d]
        return e

    def global_factor(s, ef):
        G = {}; plant_of = {src: p for p in s.plist for src in s.plants[p]["src"]}
        def g(r, depth=0):
            if r is None or depth > 20: return 0.0
            if r in G: return G[r]
            v = ef.get(r, 0.0) + g(s.plants[plant_of[r]]["dst"], depth + 1) if r in plant_of else g(CHAIN.get(r), depth + 1)
            G[r] = v; return v
        for r in s.res: g(r)
        return G

    def _model(s, price, inflow, V0, W7, MV, MF, scale=1.0, bypass="local"):
        W7 = np.asarray(W7, float) * scale
        nR = len(s.res); t = np.arange(T)
        cols, pcols = [], []
        for p in s.plist:
            d = s.plants[p]
            for src in d["src"]:
                e = 9.81e-3 * max(s.level(src, V0[s.R[src]]) - d["outlet"], 1.0) * ETA
                cols.append(dict(name=p + "|" + src, src=src, dst=d["dst"], cost=-(price - W7[s.R[src]]) * e, lb=0.0, ub=d["pmax"] / e))
                pcols.append((p, len(cols) - 1, e))
        ef = s.efactor(V0)
        fac = s.global_factor(ef) if bypass == "global" else ef
        val = lambda r: (W7[s.R[r]] * fac.get(r, 0.0)) if r else 0.0
        for nm, src, dst, cost, lo, hi in s.flows:
            wv_cost = 0.0 if lo < 0 else max(val(src) - val(dst), 0.0)
            cols.append(dict(name=nm, src=src, dst=dst, cost=np.full(T, cost + wv_cost), lb=lo, ub=hi))
        nflow = len(cols)
        mf_targets = {}
        for j, rv in enumerate(s.mf_cols):
            cols.append(dict(name="slack_" + rv, src=None, dst=None, cost=np.full(T, PEN), lb=0.0, ub=np.inf))
            mf_targets[rv] = (j, len(cols) - 1)
        NV = len(cols) * T; N = NV + nR * T
        c = np.concatenate([np.broadcast_to(cc["cost"], (T,)) for cc in cols] + [np.zeros(nR * T)])
        lb = np.concatenate([np.full(T, cc["lb"]) for cc in cols] + [np.zeros(nR * T)])
        ub = np.concatenate([np.full(T, cc["ub"]) for cc in cols] + [np.zeros(nR * T)])
        mvmap = {r: j for j, r in enumerate(s.mv_cols)}
        for r in s.res:
            i = s.R[r]; hi = max(s.maxv[i], V0[i]); sl = slice(NV + i * T, NV + (i + 1) * T)
            lo = np.minimum(MV[:, mvmap[r]], V0[i]) if r in mvmap else np.zeros(T)
            lb[sl] = np.minimum(lo, hi); ub[sl] = hi
        R_, C_, V_ = [], [], []; b = np.zeros(nR * T)
        for r in s.res:
            i = s.R[r]; rr = i * T + t
            R_ += [rr, rr[1:]]; C_ += [NV + i * T + t, NV + i * T + t[:-1]]; V_ += [np.ones(T), -np.ones(T - 1)]
            q = inflow[:, s.inflow_idx[i]] + (inflow[:, s.rvest] if r == "Bandak" else 0)
            b[rr] = DT * q; b[i * T] += V0[i]
        for k in range(nflow):
            cc = cols[k]
            if cc["src"]: R_.append(s.R[cc["src"]] * T + t); C_.append(k * T + t); V_.append(np.full(T, DT))
            if cc["dst"]: R_.append(s.R[cc["dst"]] * T + t); C_.append(k * T + t); V_.append(np.full(T, -DT))
        Aeq = sp.csr_matrix((np.concatenate(V_), (np.concatenate(R_), np.concatenate(C_))), shape=(nR * T, N))
        into_rvest = [o for o in s.out if any(to == "r_Vest_Vassdraget" for to, _ in s.out.get(o, []))]
        RI, CI, VI, bi = [], [], [], []; row = 0
        for rv, (j, sk) in mf_targets.items():
            if rv == "r_Vest_Vassdraget":
                ks = [k for k in range(nflow) if cols[k]["name"].split("#")[0] in into_rvest or cols[k]["name"].split("|")[0] == "Lio"]
                rhs = MF[:, j] - inflow[:, s.rvest]
            else:
                ks = [k for k in range(nflow) if cols[k]["name"] == rv]; rhs = MF[:, j]
            for tt in range(T):
                for k in ks: RI.append(row); CI.append(k * T + tt); VI.append(-1.0)
                RI.append(row); CI.append(sk * T + tt); VI.append(-1.0); bi.append(-rhs[tt]); row += 1
        Aub = sp.csr_matrix((VI, (RI, CI)), shape=(row, N))
        return dict(c=c, lb=lb, ub=ub, Aeq=Aeq, beq=b, Aub=Aub, bub=np.array(bi), cols=cols, pcols=pcols,
                    nflow=nflow, NV=NV, N=N, nR=nR)

    def solve(s, price, inflow, V0, W7, MV, MF, scale=1.0, bypass="local"):
        M = s._model(price, inflow, V0, W7, MV, MF, scale, bypass)
        c, cols, nflow, NV, nR = M["c"], M["cols"], M["nflow"], M["NV"], M["nR"]
        res = linprog(c, A_ub=M["Aub"], b_ub=M["bub"], A_eq=M["Aeq"], b_eq=M["beq"], bounds=np.c_[M["lb"], M["ub"]], method="highs")
        if res.status != 0:
            raise RuntimeError(res.message)
        x = res.x
        rc = c - M["Aeq"].T @ res.eqlin.marginals - M["Aub"].T @ res.ineqlin.marginals     # reduced costs
        frac, prc = {}, {}
        for p, k, e in M["pcols"]:
            frac[p] = frac.get(p, 0) + x[k * T:(k + 1) * T] / (s.plants[p]["pmax"] / e)
            r_ = rc[k * T:(k + 1) * T] / e
            prc[p] = r_ if p not in prc else np.minimum(prc[p], r_)
        spill = np.zeros((nR, T))
        for k in range(nflow):
            if cols[k]["name"].startswith("s_") and cols[k]["src"]:
                spill[s.R[cols[k]["src"]]] += x[k * T:(k + 1) * T]
        return (np.stack([np.clip(frac[p], 0, 1) for p in s.plist]).astype(np.float32),
                np.stack([prc[p] for p in s.plist]).astype(np.float32),
                (x[NV:].reshape(nR, T) / np.maximum(s.maxv, 1e-6)[:, None]).astype(np.float32),
                spill.astype(np.float32),
                (res.eqlin.marginals.reshape(nR, T) / DT).astype(np.float32))

    def solve_uc(s, price, inflow, V0, W7, MV, MF, units, groups, mip=True, node_limit=None, time_limit=None, gap=0.005, scale=1.0):
        """units: generator names (output order); groups: lists of identical generators in priority order."""
        M = s._model(price, inflow, V0, W7, MV, MF, scale)
        N0, G = M["N"], len(units); nvar = N0 + 4 * G * T; t = np.arange(T)
        off = lambda g, kind: N0 + (4 * g + kind) * T          # kind: 0 power MW, 1 on/off, 2 start, 3 stop
        c = np.r_[M["c"], np.zeros(4 * G * T)]; lb = np.r_[M["lb"], np.zeros(4 * G * T)]; ub = np.r_[M["ub"], np.zeros(4 * G * T)]
        for g, u in enumerate(units):
            info = s.gen[u]
            ub[off(g, 0):off(g, 0) + T] = info["pmax"]
            for kind in (1, 2, 3): ub[off(g, kind):off(g, kind) + T] = 1.0
            c[off(g, 2):off(g, 2) + T] = info["sc"]; c[off(g, 3):off(g, 3) + T] = info["stc"]
        # plant power = sum of its generators' power
        R_, C_, V_ = [], [], []; row = 0
        for p in s.plist:
            for pp, k, e in M["pcols"]:
                if pp == p: R_.append(row + t); C_.append(k * T + t); V_.append(np.full(T, e))
            for g, u in enumerate(units):
                if s.gen[u]["plant"] == p: R_.append(row + t); C_.append(off(g, 0) + t); V_.append(-np.ones(T))
            row += T
        A2 = sp.csr_matrix((np.concatenate(V_), (np.concatenate(R_), np.concatenate(C_))), shape=(row, nvar))
        # generator limits, start/stop logic, symmetry breaking between identical generators
        R_, C_, V_, b2 = [], [], [], []; row = 0
        def add(cols_vals, rhs, n):
            nonlocal row
            for cols_, vals in cols_vals: R_.append(row + np.arange(n)); C_.append(cols_); V_.append(vals)
            b2.append(np.full(n, rhs)); row += n
        for g, u in enumerate(units):
            info = s.gen[u]; P0, U0, S0, D0 = (off(g, k) for k in range(4))
            add([(P0 + t, np.ones(T)), (U0 + t, np.full(T, -info["pmax"]))], 0.0, T)          # P <= pmax*u
            add([(U0 + t, np.full(T, info["pmin"])), (P0 + t, -np.ones(T))], 0.0, T)          # pmin*u <= P
            t1 = t[1:]
            add([(U0 + t1, np.ones(T - 1)), (U0 + t1 - 1, -np.ones(T - 1)), (S0 + t1, -np.ones(T - 1))], 0.0, T - 1)   # start
            add([(U0 + t1 - 1, np.ones(T - 1)), (U0 + t1, -np.ones(T - 1)), (D0 + t1, -np.ones(T - 1))], 0.0, T - 1)   # stop
        for grp in groups:
            for a, b_ in zip(grp[:-1], grp[1:]):
                ga, gb = units.index(a), units.index(b_)
                add([(off(gb, 1) + t, np.ones(T)), (off(ga, 1) + t, -np.ones(T))], 0.0, T)
        A3 = sp.csr_matrix((np.concatenate(V_), (np.concatenate(R_), np.concatenate(C_))), shape=(row, nvar))
        padc = lambda A: sp.hstack([A, sp.csr_matrix((A.shape[0], nvar - A.shape[1]))])
        A = sp.vstack([padc(M["Aeq"]), A2, padc(M["Aub"]), A3]).tocsr()
        neq = M["Aeq"].shape[0] + A2.shape[0]
        lo = np.r_[M["beq"], np.zeros(A2.shape[0]), np.full(M["Aub"].shape[0] + A3.shape[0], -np.inf)]
        hi = np.r_[M["beq"], np.zeros(A2.shape[0]), M["bub"], np.concatenate(b2)]
        integ = np.zeros(nvar)
        if mip:
            for g in range(G): integ[off(g, 1):off(g, 1) + T] = 1
        opts = dict(disp=False, mip_rel_gap=gap)
        if node_limit: opts["node_limit"] = int(node_limit)
        if time_limit: opts["time_limit"] = float(time_limit)
        res = milp(c, integrality=integ, bounds=Bounds(lb, ub), constraints=LinearConstraint(A, lo, hi), options=opts)
        if res.x is None:
            return None, None, int(res.status)
        x = res.x
        U = np.stack([x[off(g, 1):off(g, 1) + T] for g in range(G)])
        Pw = np.stack([x[off(g, 0):off(g, 0) + T] / s.gen[u]["pmax"] for g, u in enumerate(units)])
        return np.clip(U, 0, 1).astype(np.float32), np.clip(Pw, 0, 1).astype(np.float32), int(res.status)


SYSTEM = System(DATA_DIR / "data/extended/Tokke_Vinje_topology.yaml", reservoirs, INFLOW_COLS, MV_COLS, MF_COLS)
PLANTS = SYSTEM.plist
assert set(SYSTEM.gen) == set(UNITS)
# identical generators, ordered by how often each was ON in the labelled weeks (symmetry breaking for the MILP)
on_rate = {u: Y[:, k].mean() for k, u in enumerate(UNITS)}
UC_GROUPS = []
for p in PLANTS:
    gl = [u for u in UNITS if SYSTEM.gen[u]["plant"] == p]
    if len(gl) > 1 and len({(SYSTEM.gen[u]["pmin"], SYSTEM.gen[u]["pmax"]) for u in gl}) == 1:
        UC_GROUPS.append(sorted(gl, key=lambda u: -on_rate[u]))
print("plants:", PLANTS); print("identical generator groups (priority order):", UC_GROUPS)

plants: ['Hogga', 'Tokke', 'Lio', 'Byrte', 'Vinje', 'Haukeli', 'Songa', 'Kjela', 'Vesle Kjela']
identical generator groups (priority order): [['Tokke_G1', 'Tokke_G3', 'Tokke_G4', 'Tokke_G2'], ['Vinje_G1', 'Vinje_G2', 'Vinje_G3']]


## Stage 1: LP simulation

In [5]:
# ======================  STAGE 1: physics simulation (LP), all variants  ======================
# Re-uses lp_outputs.npz if you uploaded it from your PC (variants already in it are not recomputed).
def _lp_batch(system, jobs):
    return [system.solve(*a) for a in jobs]

LP_FILE = RESULTS / "lp_outputs.npz"
LP = {}
for f in [LP_FILE, DATA_DIR / "lp_outputs.npz"]:
    if f.exists():
        Z = np.load(f, allow_pickle=True); pos = {r: i for i, r in enumerate(Z["run_no"])}
        if all(r in pos for r in RUN_NO):
            o = np.array([pos[r] for r in RUN_NO])
            for k in Z.files:
                if k not in ("run_no", "plants", "reservoirs") and k not in LP: LP[k] = Z[k][o]
            print("loaded", sorted(LP), "from", f)
need_base_extra = not all(k in LP for k in ("base_vol", "base_spill", "base_lam"))
t0 = time.time()
for tag, scale, bypass in LP_VARIANTS:
    if f"{tag}_frac" in LP and not (tag == "base" and need_base_extra):
        continue
    jobs = [(P[i].astype(float), I[i].astype(float), S[i].astype(float), W[i].astype(float), MV[i].astype(float),
             MF[i].astype(float), scale, bypass) for i in SEL]
    B = 32
    out = [o for b in Parallel(n_jobs=N_CPU)(delayed(_lp_batch)(SYSTEM, jobs[i:i + B]) for i in range(0, len(jobs), B)) for o in b]
    shapes = [(len(PLANTS), T), (len(PLANTS), T), (len(reservoirs), T), (len(reservoirs), T), (len(reservoirs), T)]
    full = [np.zeros((NC,) + sh, np.float32) for sh in shapes]
    for k in range(5): full[k][SEL] = np.stack([o[k] for o in out])
    LP[f"{tag}_frac"], LP[f"{tag}_rc"] = full[0], full[1]
    if tag == "base": LP["base_vol"], LP["base_spill"], LP["base_lam"] = full[2], full[3], full[4]
    np.savez_compressed(LP_FILE, run_no=RUN_NO, plants=np.array(PLANTS), reservoirs=np.array(reservoirs), **LP)
    print(f"LP variant {tag} done  [{time.time()-t0:.0f}s]", flush=True)
LP_TAGS = [t for t, _, _ in LP_VARIANTS]
print("LP variants available:", [t for t in LP_TAGS if f"{t}_frac" in LP])

loaded ['base_frac', 'base_lam', 'base_rc', 'base_spill', 'base_vol', 'g09_frac', 'g09_rc', 'g11_frac', 'g11_rc', 's08_frac', 's08_rc', 's095_frac', 's095_rc', 's09_frac', 's09_rc', 's105_frac', 's105_rc', 's11_frac', 's11_rc', 's12_frac', 's12_rc', 'v2g_frac', 'v2g_rc'] from outputs\lp_outputs.npz
LP variants available: ['base', 'v2g', 's09', 's11', 's08', 's095', 's105', 's12', 'g09', 'g11']


## Stage 2: unit-commitment MILP

In [6]:
# ======================  STAGE 2: unit-commitment MILP (on/off per generator)  ======================
# For every week and every water-value scale: the LP relaxation (fractional on/off) and the integer
# MILP solution. About 1-2 s per week per scale. Saved in chunks, so an interrupted run resumes.
UC_DIR = RESULTS / "uc_chunks"; UC_DIR.mkdir(exist_ok=True)

def _uc_chunk(system, idx, args, scale, fname):
    out = dict(idx=np.array(idx), ru=[], rp=[], mu=[], mp=[], st=[])
    for i, a in zip(idx, args):
        ru, rp, _ = system.solve_uc(*a, UNITS, UC_GROUPS, mip=False, scale=scale)
        if ru is None: ru = rp = np.zeros((len(UNITS), T), np.float32)
        mu, mp, st = None, None, -1
        if RUN_UC_MIP:
            mu, mp, st = system.solve_uc(*a, UNITS, UC_GROUPS, mip=True, node_limit=UC_NODE_LIMIT,
                                         time_limit=UC_TIME_LIMIT, gap=UC_GAP, scale=scale)
        if mu is None: mu, mp = ru, rp            # no integer solution -> fall back to the relaxation
        for k, v in zip(("ru", "rp", "mu", "mp", "st"), (ru, rp, mu, mp, st)): out[k].append(v)
    np.savez_compressed(fname, **{k: np.array(v) for k, v in out.items()})

UC = {}
t0 = time.time()
for tag, scale in UC_SCALES.items():
    CH = 16; todo = []
    for a in range(0, len(SEL), CH):
        idx = SEL[a:a + CH]; f = UC_DIR / f"{tag}_c{idx[0]:05d}_{int(RUN_UC_MIP)}_n{UC_NODE_LIMIT}.npz"
        if not f.exists():
            todo.append((idx, [(P[i].astype(float), I[i].astype(float), S[i].astype(float), W[i].astype(float),
                                MV[i].astype(float), MF[i].astype(float)) for i in idx], f))
    print(f"UC {tag} (water values x{scale}): {len(todo)} chunks to solve", flush=True)
    for g0 in range(0, len(todo), N_CPU * 4):
        grp = todo[g0:g0 + N_CPU * 4]
        Parallel(n_jobs=N_CPU)(delayed(_uc_chunk)(SYSTEM, idx, args, scale, f) for idx, args, f in grp)
        print(f"  {min(g0 + len(grp), len(todo))}/{len(todo)} chunks  [{time.time()-t0:.0f}s]", flush=True)
    d = {k: np.zeros((NC, len(UNITS), T), np.float32) for k in ("ru", "rp", "mu", "mp")}; d["st"] = np.full(NC, -2)
    for f in sorted(UC_DIR.glob(f"{tag}_c*_{int(RUN_UC_MIP)}_n{UC_NODE_LIMIT}.npz")):
        z = np.load(f)
        for k in d: d[k][z["idx"]] = z[k]
    for k, v in d.items(): UC[f"{tag}_{k}"] = v
    lab = TRAIN_ALL
    print(f"  {tag}: MILP status counts {dict(zip(*np.unique(d['st'][SEL], return_counts=True)))} | AUC of raw schedule on "
          f"labelled weeks: relaxed {roc_auc_score(Y[lab].ravel(), d['ru'][lab].ravel()):.4f}, "
          f"MILP {roc_auc_score(Y[lab].ravel(), (d['mu'] + 0.01 * d['ru'])[lab].ravel()):.4f}", flush=True)
UC_TAGS = list(UC_SCALES)

UC uc10 (water values x1.0): 0 chunks to solve
  uc10: MILP status counts {np.int64(0): np.int64(3072), np.int64(1): np.int64(575)} | AUC of raw schedule on labelled weeks: relaxed 0.9230, MILP 0.9232
UC uc09 (water values x0.9): 0 chunks to solve
  uc09: MILP status counts {np.int64(0): np.int64(3237), np.int64(1): np.int64(410)} | AUC of raw schedule on labelled weeks: relaxed 0.8675, MILP 0.8678
UC uc11 (water values x1.1): 0 chunks to solve
  uc11: MILP status counts {np.int64(0): np.int64(3058), np.int64(1): np.int64(589)} | AUC of raw schedule on labelled weeks: relaxed 0.9205, MILP 0.9208


## Stage 3: XGBoost

In [7]:
# ======================  STAGE 3: XGBoost per unit on every feature  ======================
# Features = the group's feature-engineering code (build_unit_table) + all LP variants + all MILP schedules.
# Each unit: its fixed setting from XGB_FIXED, 1.1 x the tree count found by early stopping (more training
# data than in the early-stopping run), trained on all labelled weeks, averaged over XGB_SEEDS seeds.

UPR = {"Hogga": "Bandak", "Tokke": "Vinjevatn", "Lio": "Byrtevatn", "Byrte": "Botnedalsvatn", "Vinje": "Vaamarvatn",
       "Haukeli": "Vatjern", "Songa": "Songavatn", "Kjela": "Foersvatn", "Vesle Kjela": "Kjelavatn"}
DNR = {"Hogga": None, "Tokke": "Bandak", "Lio": "Bandak", "Byrte": "Byrtevatn", "Vinje": "Vinjevatn",
       "Haukeli": "Vinjevatn", "Songa": "Totak", "Kjela": "Hyljelihyl", "Vesle Kjela": "Foersvatn"}

def shift(a, k):          # value at hour t+k (edge-padded)
    if k == 0: return a
    o = np.empty_like(a)
    if k > 0: o[:, :-k] = a[:, k:]; o[:, -k:] = a[:, -1:]
    else: o[:, -k:] = a[:, :k]; o[:, :-k] = a[:, :1]
    return o

def physics_features(unit, ci):
    pl = PLANT_OF[unit]; j = PLANTS.index(pl); k = UNITS.index(unit)
    same = [UNITS.index(u) for u in UNITS if PLANT_OF[u] == pl]
    F = {}
    # --- LP (plant level)
    fr, rc = LP["base_frac"][ci], np.clip(LP["base_rc"][ci], -500, 500)
    own = fr[:, j]
    F["lp_frac"] = own; F["lp_rc"] = rc[:, j]
    for s_ in (-3, -2, -1, 1, 2, 3): F[f"lp_frac_s{s_}"] = shift(own, s_)
    for w in (5, 13, 25): F[f"lp_frac_m{w}"] = uniform_filter1d(own, w, axis=1, mode="nearest")
    F["lp_frac_week"] = np.repeat(own.mean(1, keepdims=True), T, 1)
    for jj, p in enumerate(PLANTS):
        if jj != j: F[f"lp_frac__{p}"] = fr[:, jj]; F[f"lp_rc__{p}"] = rc[:, jj]
    u_ = reservoirs.index(UPR[pl])
    F["lp_vol_src"] = LP["base_vol"][ci][:, u_]; F["lp_spill_src"] = LP["base_spill"][ci][:, u_]
    F["lp_lam_src"] = np.clip(LP["base_lam"][ci][:, u_], -1e4, 1e4)
    if DNR[pl]:
        d_ = reservoirs.index(DNR[pl])
        F["lp_vol_dst"] = LP["base_vol"][ci][:, d_]; F["lp_lam_dst"] = np.clip(LP["base_lam"][ci][:, d_], -1e4, 1e4)
    for tg in LP_TAGS[1:]:
        f2 = LP[f"{tg}_frac"][ci]; r2 = np.clip(LP[f"{tg}_rc"][ci], -500, 500)
        F[f"lp{tg}_frac"] = f2[:, j]; F[f"lp{tg}_rc"] = r2[:, j]
        F[f"lp{tg}_frac_m5"] = uniform_filter1d(f2[:, j], 5, axis=1, mode="nearest")
        F[f"lp{tg}_frac_week"] = np.repeat(f2[:, j].mean(1, keepdims=True), T, 1)
        if tg == "v2g":
            for jj, p in enumerate(PLANTS):
                if jj != j: F[f"lp{tg}_frac__{p}"] = f2[:, jj]
    # --- unit-commitment MILP (generator level)
    for tg in UC_TAGS:
        mu, ru, mp = UC[f"{tg}_mu"][ci], UC[f"{tg}_ru"][ci], UC[f"{tg}_mp"][ci]
        F[f"{tg}_u"] = mu[:, k]; F[f"{tg}_ru"] = ru[:, k]; F[f"{tg}_p"] = mp[:, k]
        F[f"{tg}_u_week"] = np.repeat(mu[:, k].mean(1, keepdims=True), T, 1)
        F[f"{tg}_ru_week"] = np.repeat(ru[:, k].mean(1, keepdims=True), T, 1)
        F[f"{tg}_plant_n_on"] = mu[:, same].sum(1)
        if tg == "uc10":
            for s_ in (-3, -2, -1, 1, 2, 3): F[f"{tg}_u_s{s_}"] = shift(mu[:, k], s_)
            for w in (5, 13, 25): F[f"{tg}_u_m{w}"] = uniform_filter1d(mu[:, k], w, axis=1, mode="nearest")
            F[f"{tg}_status"] = np.repeat(UC[f"{tg}_st"][ci][:, None].astype(float), T, 1)
            for kk, u in enumerate(UNITS):
                if kk != k: F[f"{tg}_u__{u}"] = mu[:, kk]
    return pd.DataFrame({n: np.asarray(v, np.float32).ravel() for n, v in F.items()})

XGB_DIR = RESULTS / "xgb"; XGB_DIR.mkdir(exist_ok=True)
base_params = dict(objective="binary:logistic", tree_method="hist", device=DEVICE, subsample=0.8,
                   colsample_bytree=0.6, reg_lambda=10.0, max_bin=256, n_jobs=N_CPU)
t0 = time.time()
allc = np.sort(np.unique(np.r_[TRAIN_ALL, TEST])); where = {c: i for i, c in enumerate(allc)}
for n_u, unit in enumerate(UNITS, 1):
    f = XGB_DIR / f"{unit.replace(' ', '_')}.npy"
    if f.exists(): continue
    k = UNITS.index(unit); name, ntree = XGB_FIXED[unit]
    X = pd.concat([build_unit_table(unit, allc), physics_features(unit, allc)], axis=1)
    n_feat = X.shape[1]; X = X.to_numpy(np.float32)
    Xr = lambda ci: X[rows_of([where[c] for c in ci])]
    pt = 0
    for sd in range(XGB_SEEDS):
        m = XGBClassifier(**base_params, **XGB_CONFIGS[name], n_estimators=int(ntree * 1.1), random_state=sd)
        m.fit(Xr(TRAIN_ALL), Y[TRAIN_ALL, k].ravel())
        pt = pt + m.predict_proba(Xr(TEST))[:, 1] / XGB_SEEDS
        del m; gc.collect()
    np.save(f, pt.reshape(len(TEST), T).astype(np.float32))
    print(f"[{n_u}/14] {unit:15s} {n_feat} features | {name}, {int(ntree * 1.1)} trees  [{time.time()-t0:.0f}s]", flush=True)
    del X; gc.collect()
XGB_TEST = np.stack([np.load(XGB_DIR / f"{u.replace(' ', '_')}.npy") for u in UNITS], 1)     # (725, 14, 168)
print(f"stage 3 done; total elapsed {(time.time()-T_START)/3600:.2f} h")

stage 3 done; total elapsed 0.01 h


## Stage 4: sequence networks

In [8]:
# ======================  STAGE 4: sequence networks (GPU)  ======================
# One network sees a whole week and predicts all 14 units x 168 hours at once.
# Hourly channels: price features, per-plant price - water-value margins, water budgets, inflows,
# constraints, calendar, ALL LP variants, LP volumes/spill/shadow prices, ALL MILP schedules.
# Two architectures: dilated CNN + bidirectional GRU (as in the 0.99425 submission) and a transformer.
# Each trained on all labelled weeks (NN_SEEDS[type] seeds, averaged) and used to predict the Kaggle weeks.
UPN = {"Hogga": ["Bandak"], "Tokke": ["Vinjevatn"], "Lio": ["Byrtevatn"], "Byrte": ["Botnedalsvatn"], "Vinje": ["Vaamarvatn", "Totak"],
       "Haukeli": ["Vatjern"], "Songa": ["Songavatn", "Bitdalsvatn"], "Kjela": ["Foersvatn"], "Vesle Kjela": ["Kjelavatn"]}
slog = lambda x, s_=10.0: np.sign(x) * np.log1p(np.abs(x) / s_)
ri = reservoirs.index

def seq_inputs(ci):
    p, w7, w0, v0, inf = P[ci].astype(float), W[ci].astype(float), W0[ci].astype(float), S[ci].astype(float), I[ci].astype(float)
    n = len(ci)
    hc = [slog(p), p.argsort(1).argsort(1) / 167.0, slog(p - p.mean(1, keepdims=True))]
    for pl in PLANTS:
        wu = w7[:, [ri(x) for x in UPN[pl]]].mean(1)[:, None]; m = p - wu
        wd = w7[:, ri(DNR[pl])][:, None] if DNR[pl] else 0 * wu
        vmax = sum(capacity[ri(x)] for x in UPN[pl]); vu = sum(v0[:, ri(x)] for x in UPN[pl])[:, None]
        iu = sum(inf[:, :, INFLOW_COLS.index(x)] for x in UPN[pl]) * 3600 / 1e6
        hc += [slog(m), np.clip(m / np.maximum(np.abs(wu), 1), -5, 5), (m > 0).astype(float), slog(p - (wu - wd)),
               np.clip(np.cumsum(iu, 1) / vmax, 0, 5), np.clip((vu + np.cumsum(iu, 1)) / vmax, 0, 5)]
    t = np.broadcast_to(np.arange(T), (n, T))
    dow = (START_H[ci][:, None] // 24 + t // 24 + 3) % 7
    hc += [np.sin(2 * np.pi * t / 24), np.cos(2 * np.pi * t / 24), t / 167.0, np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7)]
    hc += [np.log1p(np.maximum(inf[:, :, j], 0)) for j in range(inf.shape[2])]
    hc += [MV[ci][:, :, j] / capacity[ri(c)] for j, c in enumerate(MV_COLS)]
    hc += [np.log1p(MF[ci][:, :, j]) for j in range(MF.shape[2])]
    for tg in LP_TAGS: hc += list(LP[f"{tg}_frac"][ci].transpose(1, 0, 2))
    for tg in LP_TAGS[:4]: hc += list(np.tanh(LP[f"{tg}_rc"][ci] / 50.0).transpose(1, 0, 2))
    hc += list(np.clip(LP["base_vol"][ci], 0, 2).transpose(1, 0, 2))
    hc += list(np.log1p(np.maximum(LP["base_spill"][ci], 0)).transpose(1, 0, 2))
    hc += list(slog(np.clip(LP["base_lam"][ci], -1e5, 1e5), 100.0).transpose(1, 0, 2))
    for tg in UC_TAGS:
        for kk in ("mu", "ru", "mp"): hc += list(UC[f"{tg}_{kk}"][ci].transpose(1, 0, 2))
    H = np.stack(hc, 1).astype(np.float32)
    st = START_H[ci].astype("datetime64[h]"); doy = (st - st.astype("datetime64[Y]")).astype(int) / 24 / 365.25
    sc = [slog(w7), slog(w7 - w0, 2.0), v0 / capacity, np.log1p(inf.sum(1) * 3600 / 1e6),
          slog(np.quantile(p, [.1, .5, .9], axis=1).T), np.stack([np.sin(2 * np.pi * doy), np.cos(2 * np.pi * doy)], 1)]
    for tg in UC_TAGS: sc.append(UC[f"{tg}_mu"][ci].mean(2))
    for tg in LP_TAGS: sc.append(LP[f"{tg}_frac"][ci].mean(2))
    return H, np.concatenate(sc, 1).astype(np.float32)

class CNNGRU(nn.Module):
    def __init__(s, ch, cs, w=160, drop=0.1):
        super().__init__()
        s.inp = nn.Conv1d(ch, w, 1); s.st = nn.Sequential(nn.Linear(cs, w), nn.GELU(), nn.Linear(w, w))
        s.blocks = nn.ModuleList([nn.Sequential(nn.Conv1d(w, w, 5, padding=2 * d, dilation=d), nn.GroupNorm(8, w), nn.GELU(),
                                                nn.Dropout(drop), nn.Conv1d(w, w, 1)) for d in (1, 2, 4, 8, 16, 1, 2, 4)])
        s.gru = nn.GRU(w, w // 2, batch_first=True, bidirectional=True)
        s.out = nn.Conv1d(2 * w, len(UNITS), 1)
    def forward(s, h, z):
        x = s.inp(h) + s.st(z)[:, :, None]
        for b in s.blocks: x = x + b(x)
        g, _ = s.gru(x.transpose(1, 2))
        return s.out(torch.cat([x, g.transpose(1, 2)], 1))

class Transformer(nn.Module):
    def __init__(s, ch, cs, d=192, layers=4, heads=8, drop=0.1):
        super().__init__()
        s.inp = nn.Conv1d(ch, d, 1); s.st = nn.Sequential(nn.Linear(cs, d), nn.GELU(), nn.Linear(d, d))
        s.local = nn.Conv1d(d, d, 5, padding=2)
        s.pos = nn.Parameter(torch.randn(1, T, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, batch_first=True, activation="gelu", norm_first=True)
        s.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        s.norm = nn.LayerNorm(d); s.out = nn.Linear(d, len(UNITS))
    def forward(s, h, z):
        x = s.inp(h) + s.st(z)[:, :, None]
        x = (x + s.local(x)).transpose(1, 2) + s.pos
        return s.out(s.norm(s.enc(x))).transpose(1, 2)

NN_CFG = {"cnn_gru": dict(cls=CNNGRU, lr=2e-3), "transformer": dict(cls=Transformer, lr=5e-4)}

def train_predict(kind, tr, pred, seed, H_all, Z_all, pos):
    torch.manual_seed(seed); np.random.seed(seed)
    itr = np.array([pos[c] for c in tr]); ipr = np.array([pos[c] for c in pred])
    Htr, Ztr = H_all[itr], Z_all[itr]
    mh, sh = Htr.mean((0, 2), keepdims=True), Htr.std((0, 2), keepdims=True) + 1e-3
    ms, ss = Ztr.mean(0), Ztr.std(0) + 1e-3
    h = torch.tensor((Htr - mh) / sh, device=DEVICE); z = torch.tensor((Ztr - ms) / ss, device=DEVICE)
    y = torch.tensor(Y[tr].astype(np.float32), device=DEVICE)
    net = NN_CFG[kind]["cls"](h.shape[1], z.shape[1]).to(DEVICE)
    lr, bs = NN_CFG[kind]["lr"], 32
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=NN_EPOCHS * int(np.ceil(len(tr) / bs)))
    lossf = nn.BCEWithLogitsLoss()
    for ep in range(NN_EPOCHS):
        net.train(); perm = torch.randperm(len(tr), device=DEVICE)
        for i in range(0, len(tr), bs):
            b = perm[i:i + bs]; opt.zero_grad(); loss = lossf(net(h[b], z[b]), y[b]); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step(); sch.step()
    hp = torch.tensor((H_all[ipr] - mh) / sh, device=DEVICE); zp = torch.tensor((Z_all[ipr] - ms) / ss, device=DEVICE)
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(ipr), 128): out.append(torch.sigmoid(net(hp[i:i + 128], zp[i:i + 128])).cpu().numpy())
    del net, h, z, y, hp, zp; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return np.concatenate(out)

NN_DIR = RESULTS / "nn"; NN_DIR.mkdir(exist_ok=True)
allc = np.sort(np.unique(np.r_[TRAIN_ALL, TEST])); pos = {c: i for i, c in enumerate(allc)}
H_ALL, Z_ALL = seq_inputs(allc)
print("sequence inputs:", H_ALL.shape[1], "hourly channels,", Z_ALL.shape[1], "weekly values", flush=True)
t0 = time.time()
NN_TEST = {}
for kind in NN_TYPES:
    preds = []
    for seed in range(NN_SEEDS[kind]):
        f = NN_DIR / f"{kind}_test_s{seed}.npy"
        if not f.exists():
            np.save(f, train_predict(kind, TRAIN_ALL, TEST, seed, H_ALL, Z_ALL, pos))
            print(f"  {kind} seed {seed} done  [{time.time()-t0:.0f}s]", flush=True)
        preds.append(np.load(f))
    NN_TEST[kind] = np.mean(preds, 0)
del H_ALL, Z_ALL; gc.collect()
print(f"stage 4 done; total elapsed {(time.time()-T_START)/3600:.2f} h")

sequence inputs: 392 hourly channels, 206 weekly values
  transformer seed 0 done  [6087s]
stage 4 done; total elapsed 1.71 h


## Stage 5: blend and submission

In [9]:
# ======================  STAGE 5: blend and submission  ======================
# final = sigmoid(0.75 logit(XGBoost) + 0.20 logit(CNN+GRU) + 0.05 logit(transformer))
L_ = lambda p: logit(np.clip(p, 1e-5, 1 - 1e-5))
MEMBERS = {"xgb": XGB_TEST, **{f"nn_{k}": v for k, v in NN_TEST.items()}}
final = expit(sum(w * L_(MEMBERS[n]) for n, w in BLEND_W.items()))
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")
sub = pd.DataFrame(final.reshape(len(TEST), -1), columns=[f"result_committed_{u}_t{h}" for u in UNITS for h in range(T)])
sub.insert(0, "Run No", RUN_NO[TEST])
sub = sub.set_index("Run No").reindex(sample_df["Run No"]).reset_index()[sample_df.columns]
if not DRY_RUN:
    assert sub.shape == sample_df.shape and np.isfinite(sub.iloc[:, 1:].to_numpy()).all()
    assert ((sub.iloc[:, 1:] >= 0) & (sub.iloc[:, 1:] <= 1)).all().all()
out_name = "submission_dry_run.csv" if DRY_RUN else "submission.csv"
sub.to_csv(DATA_DIR / out_name, index=False)
print("wrote", DATA_DIR / out_name, sub.shape)
print(f"TOTAL RUN TIME: {(time.time()-T_START)/3600:.2f} hours")

wrote submission.csv (725, 2353)
TOTAL RUN TIME: 1.71 hours
